# EN4554 Assignment 2: Convolutional Neural Networks

**Name:** ManawaduM.D  
**Index number:** 220381M  
**GitHub profile:** TODO  
**Repository:** TODO

This notebook implements Tasks 1-3 and optional Task 4. Execute the experiments to generate measured results, then complete the interpretation sections before submission.

**AI assistance acknowledgment:** OpenAI Codex assisted with notebook structure, manual differentiation code, optimizer implementations, experiment pipelines, and plotting code. Update this statement to describe all assistance used in the final submission. I am responsible for understanding and checking the submitted work.

## How to use this notebook

1. In Colab, select a T4 GPU runtime; locally, use a Python environment with PyTorch, torchvision, NumPy, and Matplotlib installed.
2. Run the setup cells. The data preparation cell downloads CIFAR-10 on its first run.
3. Run the task sections in order and review the recorded experiment settings. Full experiments take several hours; save outputs and checkpoints to Drive in Colab.
4. Add interpretations beside all plots and tables. Do not use the test set before the final evaluation in optional Task 4.
5. Save outputs, commit progress regularly, and export the completed notebook as `220381M_a02.pdf`.

All task code is implemented. Training results have not been generated locally. Review actual outputs and complete all written interpretations. Task 4 test evaluation requires RUN_FINAL_TEST=True after validation tuning.

## Persistent saving and recovery

In Colab, the setup cell mounts Google Drive. Approve the Google Drive access prompt so checkpoints and results can be saved under `MyDrive/EN4554_220381M/outputs/`.

- Training metrics, weights, optimizer/scheduler state, and random states are saved after every completed epoch. An interrupted epoch restarts from the previous saved epoch.
- After a runtime restart, run the notebook cells in order again. Completed experiments load their results; interrupted experiments continue from their saved epoch when settings match.
- Figures are also saved as PNG files. Results tables can be regenerated from saved histories without retraining.
- Save the notebook itself in Drive using Colab's **Save a copy in Drive**, and save your edits regularly. Model checkpoints do not back up notebook edits or cell output text.
- CIFAR-10 remains in runtime storage for faster access; a fresh runtime may need to download it again. Your saved experiment progress remains in Drive.
- Keep `REUSE_SAVED_RUNS = True`. If you change architecture code, change `EXPERIMENT_VERSION` to avoid reusing incompatible runs. Exact numerical reproducibility can vary across different hardware/software.



In [ ]:
# If dependencies are missing, install them in your environment first.
# Colab normally includes these packages.
from pathlib import Path
import json
import random
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
DATA_DIR = Path("data")
# Mount Drive in Colab; fail visibly rather than silently saving to temporary storage.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/EN4554_220381M/outputs')
else:
    OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(exist_ok=True)
print('Persistent results/checkpoints directory:', OUTPUT_DIR)
BATCH_SIZE = 128
NUM_WORKERS = 0  # Portable default for Windows notebooks and Colab.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print(f"PyTorch: {torch.__version__} | Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

# Atomic replacement leaves the previous checkpoint intact if writing is interrupted.
def atomic_torch_save(payload, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)

def atomic_json_save(payload, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    temporary.replace(path)

def capture_rng(loader):
    return {'python': random.getstate(), 'numpy': np.random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            'loader': loader.generator.get_state()}

def restore_rng(state, loader):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    if torch.cuda.is_available() and state['cuda'] is not None:
        if len(state['cuda']) == torch.cuda.device_count():
            torch.cuda.set_rng_state_all([s.cpu() for s in state['cuda']])
    loader.generator.set_state(state['loader'].cpu())

# Every displayed figure also gets a PNG backup on Drive.
import time as _backup_time
if not hasattr(plt, '_assignment_original_show'):
    plt._assignment_original_show = plt.show

def show_and_save(*args, **kwargs):
    for number in plt.get_fignums():
        figure = plt.figure(number)
        figure.savefig(FIGURE_DIR / f'figure_{_backup_time.time_ns()}_{number}.png',
                       dpi=150, bbox_inches='tight')
    return plt._assignment_original_show(*args, **kwargs)

plt.show = show_and_save


## Shared data setup

Use a fixed 45,000/5,000 train/validation split. Compute normalization statistics from the 45,000 training images only, before augmentation. All tasks reuse these split indices. Task 3(a) uses a fixed 5,000-image subset of the training partition.

The test dataset is deliberately not constructed here.

In [ ]:
raw_train = datasets.CIFAR10(root=DATA_DIR, train=True, download=True)

split_generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(raw_train), generator=split_generator)
train_indices = indices[:45000].tolist()
val_indices = indices[45000:].tolist()
small_train_indices = train_indices[:5000]

# Accumulate pixel moments in chunks to avoid copying the full training array.
channel_sum = np.zeros(3, dtype=np.float64)
channel_square_sum = np.zeros(3, dtype=np.float64)
pixel_count = 0
for start in range(0, len(train_indices), 1000):
    pixels = raw_train.data[train_indices[start:start + 1000]].astype(np.float64) / 255.0
    channel_sum += pixels.sum(axis=(0, 1, 2))
    channel_square_sum += np.square(pixels).sum(axis=(0, 1, 2))
    pixel_count += pixels.shape[0] * pixels.shape[1] * pixels.shape[2]
mean = channel_sum / pixel_count
std = np.sqrt(np.maximum(channel_square_sum / pixel_count - mean ** 2, 0))
del pixels

normalize = transforms.Normalize(mean.tolist(), std.tolist())
eval_transform = transforms.Compose([transforms.ToTensor(), normalize])
eval_train_dataset = datasets.CIFAR10(
    root=DATA_DIR, train=True, download=False, transform=eval_transform
)
train_dataset = Subset(eval_train_dataset, train_indices)
val_dataset = Subset(eval_train_dataset, val_indices)
small_train_dataset = Subset(eval_train_dataset, small_train_indices)

split_info = {
    "seed": SEED, "train_indices": train_indices,
    "val_indices": val_indices, "small_train_indices": small_train_indices,
    "mean": mean.tolist(), "std": std.tolist(),
}
(OUTPUT_DIR / "data_split.json").write_text(json.dumps(split_info), encoding="utf-8")
print(f"Train: {len(train_dataset)} | Validation: {len(val_dataset)}")
print("Training mean:", mean)
print("Training std:", std)

In [ ]:
def make_loader(dataset, shuffle=False, seed=SEED):
    return DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
        generator=torch.Generator().manual_seed(seed),
    )

train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset)
small_train_loader = make_loader(small_train_dataset, shuffle=True)

# Inspect training images only.
images, labels = next(iter(make_loader(train_dataset)))
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.flat, images[:10], labels[:10]):
    display_image = image.permute(1, 2, 0).numpy() * std + mean
    ax.imshow(np.clip(display_image, 0, 1))
    ax.set_title(raw_train.classes[label.item()])
    ax.axis("off")
plt.tight_layout()
plt.show()

## Shared training helpers

These helpers use autograd and ordinary cross-entropy for Tasks 2 onward. Task 1 requires a separate manual backward/training implementation. Mixup in Task 3(a) also requires adapting the training loss below.

For fair comparisons, reset the seed, recreate the model and optimizer, and recreate the shuffled loader for each run. Keep the epoch budget fixed within each comparison. Measure final training accuracy on unaugmented data, including for mixup experiments.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        logits = model(inputs)
        total_loss += nn.functional.cross_entropy(logits, targets, reduction="sum").item()
        correct += (logits.argmax(dim=1) == targets).sum().item()
        count += targets.size(0)
    return {"loss": total_loss / count, "accuracy": correct / count}


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, count = 0.0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.cross_entropy(model(inputs), targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * targets.size(0)
        count += targets.size(0)
    return total_loss / count


def fit(model, optimizer, train_loader, val_loader, epochs):
    model.to(DEVICE)
    history = []
    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        train_loss = train_epoch(model, train_loader, optimizer)
        validation = evaluate(model, val_loader)
        row = {"epoch": epoch, "train_loss": train_loss,
               "val_loss": validation["loss"], "val_accuracy": validation["accuracy"],
               "seconds": time.perf_counter() - start}
        history.append(row)
        print(f"Epoch {epoch:02d} | loss {train_loss:.4f} | "
              f"val accuracy {validation['accuracy']:.2%}")
    return history


def plot_history(history, title):
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(epochs, [r["train_loss"] for r in history], label="Train")
    axes[0].plot(epochs, [r["val_loss"] for r in history], label="Validation")
    axes[0].set(xlabel="Epoch", ylabel="Cross-entropy loss")
    axes[0].legend()
    axes[1].plot(epochs, [100 * r["val_accuracy"] for r in history])
    axes[1].set(xlabel="Epoch", ylabel="Validation accuracy (%)")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def save_results(name, config, history):
    payload = {"config": config, "history": history,
               "torch_version": torch.__version__, "seed": SEED}
    (OUTPUT_DIR / f"{name}.json").write_text(
        json.dumps(payload, indent=2), encoding="utf-8"
    )

## Task 1 - Backpropagation

### 1(a) Manual MLP

Implement `Linear`, `ReLU`, and numerically stable softmax cross-entropy with explicit forward/backward methods. Cache the inputs needed for the backward pass. Assemble an MLP accepting flattened CIFAR-10 images (3,072 inputs, 10 output logits). Do not use autograd for the manual implementation.

In [ ]:
class ManualLinear:
    def __init__(self, inputs, outputs, dtype=torch.float32, device="cpu"):
        self.weight = torch.randn(inputs, outputs, dtype=dtype, device=device) * (2 / inputs) ** 0.5
        self.bias = torch.zeros(outputs, dtype=dtype, device=device)
    def forward(self, x):
        self.x = x
        return x @ self.weight + self.bias
    def backward(self, gradient):
        self.dw = self.x.T @ gradient
        self.db = gradient.sum(0)
        return gradient @ self.weight.T

class ManualReLU:
    def forward(self, x):
        self.mask = x > 0
        return x.clamp_min(0)
    def backward(self, gradient):
        return gradient * self.mask

class ManualSigmoid:
    def forward(self, x):
        self.output = torch.sigmoid(x)
        return self.output
    def backward(self, gradient):
        return gradient * self.output * (1 - self.output)

class ManualCrossEntropy:
    def forward(self, logits, labels):
        log_probabilities = logits - torch.logsumexp(logits, dim=1, keepdim=True)
        self.probabilities = log_probabilities.exp()
        self.labels = labels
        return -log_probabilities[torch.arange(len(labels), device=labels.device), labels].mean()
    def backward(self):
        gradient = self.probabilities.clone()
        gradient[torch.arange(len(self.labels), device=self.labels.device), self.labels] -= 1
        return gradient / len(self.labels)

class ManualMLP:
    def __init__(self, widths, activation="relu", dtype=torch.float32, device="cpu"):
        self.layers = []
        for i, (inputs, outputs) in enumerate(zip(widths[:-1], widths[1:])):
            self.layers.append(ManualLinear(inputs, outputs, dtype, device))
            if i < len(widths) - 2:
                self.layers.append(ManualReLU() if activation == "relu" else ManualSigmoid())
        self.criterion = ManualCrossEntropy()
    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x
    def loss(self, x, y):
        return self.criterion.forward(self.forward(x), y)
    def backward(self):
        gradient = self.criterion.backward()
        for layer in reversed(self.layers):
            gradient = layer.backward(gradient)
        return gradient
    def linears(self):
        return [layer for layer in self.layers if isinstance(layer, ManualLinear)]
    def parameters_and_gradients(self):
        return [(p, g) for layer in self.linears()
                for p, g in [(layer.weight, layer.dw), (layer.bias, layer.db)]]

print("Manual layers ready; no autograd is used by these components.")


### 1(b) Gradient verification

Use a small deterministic example and float64 precision. Compare every parameter gradient and the input gradient with autograd and centred finite differences:

`df/dx_i ~= [f(x + h e_i) - f(x - h e_i)] / (2h)`.

Report relative errors, your definition of relative error, the finite-difference step size, and `torch.allclose` results with explicit tolerances. Use autograd only as the reference check.

In [ ]:
seed_everything()
check_model = ManualMLP([4, 5, 3], dtype=torch.float64)
check_x = torch.randn(3, 4, dtype=torch.float64)
check_y = torch.tensor([0, 1, 2])
manual_loss = check_model.loss(check_x, check_y)
manual_dx = check_model.backward().clone()
pairs = check_model.parameters_and_gradients()
manual_gradients = [g.clone() for _, g in pairs] + [manual_dx]
reference_parameters = [p.clone().requires_grad_(True) for p, _ in pairs]
reference_x = check_x.clone().requires_grad_(True)
hidden = torch.relu(reference_x @ reference_parameters[0] + reference_parameters[1])
reference_loss = nn.functional.cross_entropy(
    hidden @ reference_parameters[2] + reference_parameters[3], check_y)
reference_loss.backward()
autograd_gradients = [p.grad for p in reference_parameters] + [reference_x.grad]

def relative_error(a, b):
    return ((a-b).norm() / (a.norm()+b.norm()).clamp_min(1e-15)).item()

step = 1e-6
names = ["W1", "b1", "W2", "b2", "input"]
print("Relative error = ||a-b|| / (||a|| + ||b||); h=1e-6")
print(f"{'Tensor':8s} {'Autograd error':>16s} {'Finite-diff error':>18s}  allclose(auto, FD)")
for name, tensor, manual, reference in zip(
        names, [p for p, _ in pairs] + [check_x], manual_gradients, autograd_gradients):
    numerical = torch.zeros_like(tensor)
    for j in range(tensor.numel()):
        flat = tensor.view(-1)
        original = flat[j].item()
        flat[j] = original + step
        plus = check_model.loss(check_x, check_y).item()
        flat[j] = original - step
        minus = check_model.loss(check_x, check_y).item()
        flat[j] = original
        numerical.view(-1)[j] = (plus - minus) / (2 * step)
    auto_ok = torch.allclose(manual, reference, rtol=1e-5, atol=1e-7)
    fd_ok = torch.allclose(manual, numerical, rtol=1e-5, atol=1e-7)
    print(f"{name:8s} {relative_error(manual, reference):16.3e} "
          f"{relative_error(manual, numerical):18.3e}  {auto_ok}, {fd_ok}")
    assert auto_ok and fd_ok, f"Gradient mismatch: {name}"


### 1(c) Train the manual MLP

Train with manual SGD updates on flattened CIFAR-10 images. Plot loss and report validation accuracy. Record learning rate, batch size, epoch count, and initialization.

In [ ]:
seed_everything()
MLP_EPOCHS = 20
MLP_LR = 0.01
manual_model = ManualMLP([3072, 256, 10], device=DEVICE)
manual_train_loader = make_loader(train_dataset, shuffle=True)
manual_history = []

@torch.no_grad()
def evaluate_manual(model, loader):
    loss_sum, correct, count = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE).flatten(1), y.to(DEVICE)
        logits = model.forward(x)
        loss_sum += nn.functional.cross_entropy(logits, y, reduction="sum").item()
        correct += (logits.argmax(1) == y).sum().item()
        count += len(y)
    return {"loss": loss_sum / count, "accuracy": correct / count}

manual_config = {'widths': [3072, 256, 10], 'lr': MLP_LR, 'epochs': MLP_EPOCHS,
                 'batch_size': BATCH_SIZE, 'seed': SEED,
                 'mean': mean.tolist(), 'std': std.tolist()}
manual_resume_path = OUTPUT_DIR / 'task1_mlp.resume.pt'
if manual_resume_path.exists():
    saved = torch.load(manual_resume_path, map_location='cpu', weights_only=False)
    if saved['config'] == manual_config:
        for layer, state in zip(manual_model.linears(), saved['layers']):
            layer.weight.copy_(state['weight'].to(DEVICE))
            layer.bias.copy_(state['bias'].to(DEVICE))
        manual_history = saved['history']
        restore_rng(saved['rng'], manual_train_loader)
        print(f'Task 1 MLP: resuming after epoch {len(manual_history)}')

for epoch in range(len(manual_history) + 1, MLP_EPOCHS + 1):
    loss_sum, count = 0.0, 0
    for x, y in manual_train_loader:
        x, y = x.to(DEVICE).flatten(1), y.to(DEVICE)
        loss = manual_model.loss(x, y)
        manual_model.backward()
        for parameter, gradient in manual_model.parameters_and_gradients():
            parameter.add_(gradient, alpha=-MLP_LR)
        loss_sum += loss.item() * len(y)
        count += len(y)
    metrics = evaluate_manual(manual_model, val_loader)
    manual_history.append({"epoch": epoch, "train_loss": loss_sum / count,
                           "val_loss": metrics["loss"], "val_accuracy": metrics["accuracy"]})
    print(f"Epoch {epoch:02d}/{MLP_EPOCHS}: loss={loss_sum/count:.4f}, "
          f"validation accuracy={metrics['accuracy']:.2%}")
    atomic_torch_save({
        'config': manual_config,
        'layers': [{'weight': layer.weight.detach().cpu(), 'bias': layer.bias.detach().cpu()}
                   for layer in manual_model.linears()],
        'history': manual_history, 'rng': capture_rng(manual_train_loader),
    }, manual_resume_path)
    atomic_json_save({'config': manual_config, 'history': manual_history},
                     OUTPUT_DIR / 'task1_mlp.json')
plot_history(manual_history, "Task 1(c): manual MLP")
save_results("task1_mlp", {"widths": [3072, 256, 10], "lr": MLP_LR,
                           "epochs": MLP_EPOCHS, "batch_size": BATCH_SIZE}, manual_history)
print(f"Final validation accuracy: {manual_history[-1]['val_accuracy']:.2%}")


### 1(d) Gradient flow in a 10-layer MLP

Define clearly how you count the 10 layers. At initialization, run forward/backward on a fixed batch for sigmoid and ReLU networks. Plot per-layer gradient norms and record the initialization used so that the comparison is interpretable.

In [ ]:
# Ten linear layers: nine hidden layers and one output layer.
# Identical weights and Kaiming-normal initialization isolate activation effects.
seed_everything()
gradient_x, gradient_y = next(iter(make_loader(train_dataset)))
gradient_x = gradient_x.to(DEVICE).flatten(1)
gradient_y = gradient_y.to(DEVICE)
widths = [3072] + [128] * 9 + [10]
gradient_results = {}
for activation in ["sigmoid", "relu"]:
    seed_everything()
    network = ManualMLP(widths, activation=activation, device=DEVICE)
    network.loss(gradient_x, gradient_y)
    network.backward()
    norms = [layer.dw.norm().item() for layer in network.linears()]
    gradient_results[activation] = norms
    print(activation, "weight-gradient norms:", [f"{v:.3e}" for v in norms])
    plt.semilogy(range(1, 11), np.maximum(norms, 1e-30), marker="o", label=activation)
plt.xticks(range(1, 11))
plt.xlabel("Linear layer (input to output)")
plt.ylabel("Weight-gradient L2 norm (log scale)")
plt.title("Task 1(d): gradient norms at initialization")
plt.legend()
plt.tight_layout()
plt.show()
(OUTPUT_DIR / "task1_gradient_norms.json").write_text(json.dumps(gradient_results, indent=2))


**Task 1 interpretation - TODO:** Explain the gradient checks, training behavior, and how/why gradient magnitudes change with depth.

## Task 2 - Optimizers

### 2(a) Custom momentum SGD and Adam

Implement both as `torch.optim.Optimizer` subclasses. Include Adam bias correction. Compare several consecutive updates against `torch.optim.SGD` and `torch.optim.Adam`, starting from identical parameters and using identical gradients and hyperparameters. Report numerical errors and matching checks.

In [ ]:
# %% Task 2(a): custom optimizers and numerical verification
import math
import copy
import hashlib
from IPython.display import display

class MomentumSGD(torch.optim.Optimizer):
    """SGD with classical momentum, optional coupled weight decay, no dampening."""
    def __init__(self, params, lr=0.01, momentum=0.9, weight_decay=0.0):
        if lr < 0 or not 0 <= momentum < 1 or weight_decay < 0:
            raise ValueError("Invalid optimizer hyperparameters")
        super().__init__(params, dict(lr=lr, momentum=momentum, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                if p.grad.is_sparse:
                    raise RuntimeError('Sparse gradients are unsupported')
                gradient = p.grad
                if group['weight_decay']:
                    gradient = gradient.add(p, alpha=group['weight_decay'])
                if group['momentum']:
                    state = self.state[p]
                    if 'velocity' not in state:
                        state['velocity'] = gradient.clone()
                    else:
                        state['velocity'].mul_(group['momentum']).add_(gradient)
                    gradient = state['velocity']
                p.add_(gradient, alpha=-group['lr'])
        return loss

class CustomAdam(torch.optim.Optimizer):
    """Adam with bias correction and optional coupled L2 weight decay (not AdamW)."""
    def __init__(self, params, lr=0.001, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        if lr < 0 or eps < 0 or weight_decay < 0 or any(not 0 <= b < 1 for b in betas):
            raise ValueError('Invalid optimizer hyperparameters')
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            beta1, beta2 = group['betas']
            for p in group['params']:
                if p.grad is None:
                    continue
                if p.grad.is_sparse:
                    raise RuntimeError('Sparse gradients are unsupported')
                gradient = p.grad
                if group['weight_decay']:
                    gradient = gradient.add(p, alpha=group['weight_decay'])
                state = self.state[p]
                if not state:
                    state.update(step=0, m=torch.zeros_like(p), v=torch.zeros_like(p))
                state['step'] += 1
                state['m'].mul_(beta1).add_(gradient, alpha=1-beta1)
                state['v'].mul_(beta2).addcmul_(gradient, gradient, value=1-beta2)
                m_hat = state['m'] / (1-beta1**state['step'])
                v_hat = state['v'] / (1-beta2**state['step'])
                p.addcdiv_(m_hat, v_hat.sqrt().add_(group['eps']), value=-group['lr'])
        return loss

# Ten deterministic updates, two parameter groups, a missing-gradient step,
# and both zero/nonzero weight decay. Float64 exposes numerical differences.
optimizer_check_rows = []
for weight_decay in (0.0, 0.01):
    for name, implementation, reference in [
        ('Momentum', MomentumSGD, torch.optim.SGD),
        ('Adam', CustomAdam, torch.optim.Adam),
    ]:
        seed_everything()
        parameters = [nn.Parameter(torch.randn(3, 4, dtype=torch.float64)),
                      nn.Parameter(torch.randn(4, dtype=torch.float64))]
        references = [nn.Parameter(p.detach().clone()) for p in parameters]
        kwargs = {'lr': 0.007, 'weight_decay': weight_decay}
        kwargs.update({'momentum': 0.85} if name == 'Momentum' else {'betas': (0.8, 0.95)})
        opt = implementation([{'params': [parameters[0]]},
                              {'params': [parameters[1]], 'lr': 0.003}], **kwargs)
        ref = reference([{'params': [references[0]]},
                         {'params': [references[1]], 'lr': 0.003}], foreach=False, **kwargs)
        max_error = 0.0
        matched = True
        for step in range(10):
            for j, (p, q) in enumerate(zip(parameters, references)):
                gradient = None if step == 4 and j == 1 else torch.randn_like(p)
                p.grad = gradient
                q.grad = None if gradient is None else gradient.clone()
            opt.step()
            ref.step()
            for p, q in zip(parameters, references):
                max_error = max(max_error, (p-q).abs().max().item())
                matched &= torch.allclose(p, q, rtol=1e-10, atol=1e-12)
        optimizer_check_rows.append([name, weight_decay, max_error, matched])
        assert matched, f'{name} did not match PyTorch'
print('Optimizer | weight decay | max absolute parameter error | allclose')
for row in optimizer_check_rows:
    print(f'{row[0]:8s} | {row[1]:.2f} | {row[2]:.3e} | {row[3]}')



### 2(b) Optimizer trajectories

Plot contours of `f(x, y) = 0.5 * (x**2 + 50*y**2)` and the trajectories of plain SGD, momentum SGD, and Adam. Its gradient is `(x, 50*y)`. Record the common starting point, learning rates, and iteration count.

In [ ]:
# %% Task 2(b): trajectories on an ill-conditioned quadratic
quadratic_paths = {}
quadratic_settings = {'SGD': 0.03, 'Momentum': 0.03, 'Adam': 0.1}
for name, lr in quadratic_settings.items():
    position = nn.Parameter(torch.tensor([3.0, 3.0], dtype=torch.float64))
    optimizer = (torch.optim.SGD([position], lr=lr) if name == 'SGD' else
                 MomentumSGD([position], lr=lr, momentum=0.9) if name == 'Momentum' else
                 CustomAdam([position], lr=lr))
    path = [position.detach().numpy().copy()]
    for _ in range(150):
        optimizer.zero_grad(set_to_none=True)
        objective = 0.5 * (position[0]**2 + 50*position[1]**2)
        objective.backward()
        optimizer.step()
        path.append(position.detach().numpy().copy())
    quadratic_paths[name] = np.asarray(path)

xx, yy = np.meshgrid(np.linspace(-3.5, 3.5, 300), np.linspace(-3.5, 3.5, 300))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].contour(xx, yy, 0.5*(xx**2+50*yy**2), levels=np.geomspace(0.05, 300, 18))
for name, path in quadratic_paths.items():
    axes[0].plot(path[:, 0], path[:, 1], '.-', markersize=2,
                 label=f'{name}, lr={quadratic_settings[name]}')
    values = 0.5*(path[:, 0]**2 + 50*path[:, 1]**2)
    axes[1].semilogy(np.maximum(values, 1e-20), label=name)
    print(f'{name}: final objective={values[-1]:.6g}, position={path[-1]}')
axes[0].set(xlabel='x', ylabel='y', title='Starting point (3, 3); 150 updates')
axes[1].set(xlabel='Update', ylabel='Objective (log scale)', title='Convergence')
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()



### 2(c) Learning-rate sensitivity on a small CNN

Use one fixed CNN and three learning rates for each of plain SGD, momentum SGD, and Adam (nine runs). Select each optimizer's best rate using validation results. Plot validation accuracy versus learning rate and training loss at the best rate. Record the selection rule and use a consistent training budget.

In [ ]:
# %% Task 2(c): shared experiment engine and nine learning-rate runs
# OUTPUT_DIR is configured by the setup cell (Google Drive in Colab).
EPOCHS_T2 = 10
EPOCHS_T3A = 30
EPOCHS_T3B = 30
EPOCHS_T4 = 20
REUSE_SAVED_RUNS = True
EXPERIMENT_VERSION = '220381M-v1'

class SmallCNN(nn.Module):
    def __init__(self, batch_norm=False, dropout=0.0):
        super().__init__()
        layers = []
        channels = 3
        for width in (32, 64, 128):
            layers.append(nn.Conv2d(channels, width, 3, padding=1, bias=not batch_norm))
            if batch_norm:
                layers.append(nn.BatchNorm2d(width))
            layers.extend([nn.ReLU(), nn.MaxPool2d(2)])
            channels = width
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(dropout), nn.Linear(128, 10))
        initialize_network(self)
    def forward(self, x):
        return self.head(self.features(x))

def initialize_network(model):
    for layer in model.modules():
        if isinstance(layer, nn.Conv2d):
            nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)
        elif isinstance(layer, nn.BatchNorm2d):
            nn.init.ones_(layer.weight)
            nn.init.zeros_(layer.bias)
        elif isinstance(layer, nn.Linear):
            nn.init.kaiming_normal_(layer.weight, nonlinearity='linear')
            nn.init.zeros_(layer.bias)

def make_optimizer(name, parameters, lr, weight_decay=0.0):
    if name == 'SGD':
        return torch.optim.SGD(parameters, lr=lr, weight_decay=weight_decay)
    if name == 'Momentum':
        return MomentumSGD(parameters, lr=lr, momentum=0.9, weight_decay=weight_decay)
    if name == 'Adam':
        return CustomAdam(parameters, lr=lr, weight_decay=weight_decay)
    raise ValueError(name)

def show_table(rows, columns):
    # Render an actual table in notebook/PDF, with no pandas dependency.
    from html import escape
    header = ''.join(f'<th>{escape(str(c))}</th>' for c in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(v))}</td>' for v in row) + '</tr>'
                   for row in rows)
    display(__import__('IPython').display.HTML(
        '<table><thead><tr>'+header+'</tr></thead><tbody>'+body+'</tbody></table>'))

def run_experiment(tag, model_factory, config, training_dataset=train_dataset,
                   clean_dataset=train_dataset, augment_batch=None, mixup_alpha=0.0):
    """Save every completed epoch and resume interrupted runs; never select using test data."""
    signature = {**config, 'seed': SEED, 'batch_size': BATCH_SIZE,
                 'train_size': len(training_dataset), 'version': EXPERIMENT_VERSION,
                 'mean': mean.tolist(), 'std': std.tolist()}
    digest = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:12]
    stem = OUTPUT_DIR / f'{tag}_{digest}'
    result_path = Path(str(stem) + '.json')
    checkpoint_path = Path(str(stem) + '.pt')
    resume_path = Path(str(stem) + '.resume.pt')
    if REUSE_SAVED_RUNS and result_path.exists() and checkpoint_path.exists():
        result = json.loads(result_path.read_text())
        if result['config'] == signature and result['status'] == 'complete':
            print(f'{tag}: loaded saved run')
            return result
    seed_everything()
    model = model_factory().to(DEVICE)
    optimizer = make_optimizer(config['optimizer'], model.parameters(), config['lr'],
                               config.get('weight_decay', 0.0))
    loader = make_loader(training_dataset, shuffle=True)
    clean_loader = make_loader(clean_dataset)
    scheduler = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, config['epochs'])
                 if config.get('cosine', False) else None)
    history = []
    elapsed_before = 0.0
    if REUSE_SAVED_RUNS and resume_path.exists():
        # Load only checkpoints created by this notebook; RNG state includes Python objects.
        saved = torch.load(resume_path, map_location='cpu', weights_only=False)
        if saved['config'] == signature:
            model.load_state_dict(saved['model'])
            optimizer.load_state_dict(saved['optimizer'])
            if scheduler is not None:
                scheduler.load_state_dict(saved['scheduler'])
            history = saved['history']
            elapsed_before = saved['seconds']
            restore_rng(saved['rng'], loader)
            print(f'{tag}: resuming after epoch {len(history)}')
    started = time.perf_counter()
    status = 'complete'
    for epoch in range(len(history) + 1, config['epochs'] + 1):
        model.train()
        loss_sum, seen = 0.0, 0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if augment_batch is not None:
                x = augment_batch(x)
            optimizer.zero_grad(set_to_none=True)
            if mixup_alpha > 0:
                x, targets_a, targets_b, lam = mixup_batch(x, y, mixup_alpha)
                logits = model(x)
                loss = (lam * nn.functional.cross_entropy(logits, targets_a) +
                        (1-lam) * nn.functional.cross_entropy(logits, targets_b))
            else:
                loss = nn.functional.cross_entropy(model(x), y)
            if not torch.isfinite(loss):
                status = 'diverged'
                break
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(y)
            seen += len(y)
        if status != 'complete' or not all(torch.isfinite(p).all() for p in model.parameters()):
            status = 'diverged'
            print(f'{tag}: non-finite loss/parameters at epoch {epoch}; excluded from selection')
            break
        validation = evaluate(model, val_loader)
        clean_train = evaluate(model, clean_loader)
        if not all(math.isfinite(v) for v in [validation['loss'], clean_train['loss']]):
            status = 'diverged'
            break
        history.append({'epoch': epoch, 'train_loss': loss_sum/seen,
                        'clean_train_loss': clean_train['loss'],
                        'train_accuracy': clean_train['accuracy'],
                        'val_loss': validation['loss'], 'val_accuracy': validation['accuracy']})
        print(f"{tag} {epoch:02d}/{config['epochs']}: loss={loss_sum/seen:.4f}, "
              f"train={clean_train['accuracy']:.2%}, val={validation['accuracy']:.2%}")
        if scheduler is not None:
            scheduler.step()
        # Commit a complete epoch, including optimizer, scheduler and random states.
        atomic_torch_save({
            'config': signature, 'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict() if scheduler is not None else None,
            'history': history, 'seconds': elapsed_before + time.perf_counter()-started,
            'rng': capture_rng(loader),
        }, resume_path)
        atomic_json_save({'config': signature, 'history': history,
                          'status': 'in_progress'}, result_path)
    result = {'config': signature, 'history': history, 'status': status,
              'parameters': sum(p.numel() for p in model.parameters()),
              'seconds': elapsed_before + time.perf_counter()-started, 'checkpoint': str(checkpoint_path)}
    if status == 'complete':
        atomic_torch_save({k: v.detach().cpu() for k, v in model.state_dict().items()}, checkpoint_path)
    atomic_json_save(result, result_path)
    return result

def comparison_table(results):
    rows = []
    for name, run in results.items():
        if run['status'] == 'complete':
            r = run['history'][-1]
            rows.append([name, f"{100*r['train_accuracy']:.2f}",
                         f"{100*r['val_accuracy']:.2f}",
                         f"{100*(r['train_accuracy']-r['val_accuracy']):.2f}",
                         f"{r['train_loss']:.4f}", run['parameters'], f"{run['seconds']/60:.1f}"])
        else:
            rows.append([name, 'DIVERGED', '-', '-', '-', run['parameters'], '-'])
    show_table(rows, ['Configuration', 'Clean train (%)', 'Validation (%)',
                      'Gap (pp)', 'Training loss', 'Parameters', 'Minutes'])

def plot_comparison(results, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for name, run in results.items():
        h = run['history']
        if not h:
            continue
        epochs = [r['epoch'] for r in h]
        axes[0].plot(epochs, [r['train_loss'] for r in h], label=name)
        line, = axes[1].plot(epochs, [100*r['val_accuracy'] for r in h], label=name+' val')
        axes[1].plot(epochs, [100*r['train_accuracy'] for r in h], '--',
                     color=line.get_color(), alpha=0.6, label=name+' train')
    axes[0].set(xlabel='Epoch', ylabel='Training objective')
    axes[1].set(xlabel='Epoch', ylabel='Accuracy (%)')
    for ax in axes:
        ax.legend(fontsize=7)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

LEARNING_RATES = {'SGD': [0.01, 0.05, 0.1],
                  'Momentum': [0.005, 0.02, 0.05],
                  'Adam': [0.0001, 0.001, 0.01]}
task2_results = {}
for name, rates in LEARNING_RATES.items():
    for lr in rates:
        tag = f'task2_{name}_{lr}'
        config = {'architecture': 'SmallCNN_noBN', 'optimizer': name, 'lr': lr,
                  'epochs': EPOCHS_T2, 'selection': 'final_validation_accuracy'}
        task2_results[(name, lr)] = run_experiment(tag, SmallCNN, config)
comparison_table({f'{name}, lr={lr}': r for (name, lr), r in task2_results.items()})
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
task2_best = {}
for name, rates in LEARNING_RATES.items():
    eligible = [(lr, task2_results[(name, lr)]) for lr in rates
                if task2_results[(name, lr)]['status'] == 'complete']
    axes[0].semilogx([lr for lr, _ in eligible],
                    [100*r['history'][-1]['val_accuracy'] for _, r in eligible], 'o-', label=name)
    if eligible:
        best_lr, best = max(eligible, key=lambda item: item[1]['history'][-1]['val_accuracy'])
        task2_best[name] = best_lr
        axes[1].plot([r['epoch'] for r in best['history']],
                     [r['train_loss'] for r in best['history']], label=f'{name}, lr={best_lr}')
axes[0].set(xlabel='Learning rate (log scale)', ylabel='Final validation accuracy (%)')
axes[1].set(xlabel='Epoch', ylabel='Training loss at best learning rate')
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()
print('Best rates selected using final validation accuracy:', task2_best)



**Task 2 interpretation - TODO:** Explain trajectory differences, optimizer sensitivity, and what your measured CNN results support.

## Task 3 - Regularization and residual connections

### 3(a) Regularization on 5,000 training images

Use the fixed `small_train_indices` and a small CNN with batch normalization. Compare baseline, weight decay, dropout, flip+crop, cutout, mixup, and all combined. Implement cutout and mixup yourself.

Use separate dataset instances for training augmentation and clean accuracy measurement. Keep the validation set unchanged. Report clean training accuracy, validation accuracy, and their gap (in percentage points); plot training and validation curves. Document augmentation order and all hyperparameters.

In [ ]:
# %% Task 3(a): seven controlled regularization experiments
class Cutout:
    """Mask a randomly centred square in each normalized tensor; zero is channel mean."""
    def __init__(self, length=8):
        self.length = length
    def __call__(self, image):
        image = image.clone()
        height, width = image.shape[-2:]
        y = torch.randint(height, ()).item()
        x = torch.randint(width, ()).item()
        half = self.length // 2
        image[:, max(0, y-half):min(height, y-half+self.length),
              max(0, x-half):min(width, x-half+self.length)] = 0
        return image

def mixup_batch(inputs, labels, alpha=0.2):
    lam = float(np.random.beta(alpha, alpha))
    permutation = torch.randperm(len(inputs), device=inputs.device)
    return lam*inputs + (1-lam)*inputs[permutation], labels, labels[permutation], lam

def augmented_dataset(indices, flip_crop=False, cutout=False):
    operations = []
    if flip_crop:
        operations += [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()]
    operations += [transforms.ToTensor(), normalize]
    if cutout:
        operations.append(Cutout(8))
    dataset = datasets.CIFAR10(root=DATA_DIR, train=True, download=False,
                               transform=transforms.Compose(operations))
    return Subset(dataset, indices)

regularization_configs = {
    'baseline': {}, 'weight_decay': {'weight_decay': 5e-4},
    'dropout': {'dropout': 0.3}, 'flip_crop': {'flip_crop': True},
    'cutout': {'cutout': True}, 'mixup': {'mixup_alpha': 0.2},
    'all_combined': {'weight_decay': 5e-4, 'dropout': 0.3, 'flip_crop': True,
                     'cutout': True, 'mixup_alpha': 0.2},
}
# Display augmentation examples using training data only.
seed_everything()
preview_data = augmented_dataset(small_train_indices, flip_crop=True, cutout=True)
fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for j, ax in enumerate(axes):
    x, y = preview_data[j]
    ax.imshow(np.clip(x.permute(1, 2, 0).numpy()*std+mean, 0, 1))
    ax.set_title(raw_train.classes[y])
    ax.axis('off')
plt.suptitle('Task 3(a): flip/crop + custom cutout examples')
plt.tight_layout()
plt.show()

task3a_results = {}
for name, settings in regularization_configs.items():
    config = {'architecture': 'SmallCNN_BN', 'optimizer': 'Momentum', 'lr': 0.03,
              'epochs': EPOCHS_T3A, **settings}
    data = augmented_dataset(small_train_indices, settings.get('flip_crop', False),
                             settings.get('cutout', False))
    task3a_results[name] = run_experiment(
        'task3a_'+name,
        lambda settings=settings: SmallCNN(batch_norm=True, dropout=settings.get('dropout', 0.0)),
        config, training_dataset=data, clean_dataset=small_train_dataset,
        mixup_alpha=settings.get('mixup_alpha', 0.0))
comparison_table(task3a_results)
# Split curves into readable groups rather than placing fourteen lines in one panel.
for names in [('baseline', 'weight_decay', 'dropout', 'flip_crop'),
              ('baseline', 'cutout', 'mixup', 'all_combined')]:
    plot_comparison({name: task3a_results[name] for name in names}, 'Task 3(a): regularization')



**Task 3(a) interpretation - TODO:** Discuss overfitting, the accuracy gap, and whether combining regularizers improves your measured results.

### 3(b) Plain versus residual networks

Implement depths `6n + 2` = 8, 20, 56 (`n` = 1, 3, 9), using three stages with channels 16/32/64, batch normalization, Kaiming initialization, and global average pooling. Within each matched pair, the only architectural difference must be the shortcut. Document handling of shortcuts when spatial size or channel count changes.

Train all six models with a consistent protocol and plot training loss versus epoch.

In [ ]:
# %% Task 3(b): matched CIFAR plain/residual families, depths 8/20/56
class CifarBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, residual=True):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.residual = residual
        self.stride = stride
        self.out_channels = out_channels
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.residual:
            shortcut = x[:, :, ::self.stride, ::self.stride]
            # Parameter-free option-A shortcut: spatial subsampling and zero padding.
            difference = self.out_channels - shortcut.shape[1]
            if difference:
                left = difference // 2
                shortcut = nn.functional.pad(shortcut, (0, 0, 0, 0, left, difference-left))
            out = out + shortcut
        return torch.relu(out)

class CifarNetwork(nn.Module):
    def __init__(self, depth=20, residual=True, widths=(16, 32, 64)):
        super().__init__()
        if depth < 8 or (depth-2) % 6:
            raise ValueError('Depth must equal 6n+2 with n >= 1')
        blocks_per_stage = (depth-2)//6
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, padding=1, bias=False),
                                  nn.BatchNorm2d(widths[0]), nn.ReLU())
        blocks = []
        channels = widths[0]
        for stage, width in enumerate(widths):
            for block in range(blocks_per_stage):
                stride = 2 if stage > 0 and block == 0 else 1
                blocks.append(CifarBlock(channels, width, stride, residual))
                channels = width
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(channels, 10))
        initialize_network(self)
    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))

# Verify depth counts and parameter equality before any training.
for depth in (8, 20, 56):
    plain = CifarNetwork(depth, False)
    residual = CifarNetwork(depth, True)
    assert sum(isinstance(m, (nn.Conv2d, nn.Linear)) for m in plain.modules()) == depth
    assert sum(p.numel() for p in plain.parameters()) == sum(p.numel() for p in residual.parameters())
    print(f'Depth {depth}: {sum(p.numel() for p in plain.parameters()):,} parameters in each family')
del plain, residual

task3b_results = {}
# Same augmentation, optimizer, initialization seed and training budget for all six runs.
depth_train_dataset = augmented_dataset(train_indices, flip_crop=True)
for depth in (8, 20, 56):
    for residual in (False, True):
        name = f'{"residual" if residual else "plain"}_{depth}'
        config = {'architecture': 'CifarNetwork', 'depth': depth, 'residual': residual,
                  'widths': [16, 32, 64], 'optimizer': 'Momentum', 'lr': 0.05,
                  'weight_decay': 5e-4, 'cosine': True, 'flip_crop': True,
                  'epochs': EPOCHS_T3B}
        task3b_results[name] = run_experiment(
            'task3b_'+name, lambda d=depth, r=residual: CifarNetwork(d, r), config,
            training_dataset=depth_train_dataset)
comparison_table(task3b_results)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, depth in zip(axes, (8, 20, 56)):
    for family in ('plain', 'residual'):
        history = task3b_results[f'{family}_{depth}']['history']
        ax.plot([r['epoch'] for r in history], [r['train_loss'] for r in history], label=family)
    ax.set(title=f'Depth {depth}', xlabel='Epoch', ylabel='Training loss')
    ax.legend()
plt.suptitle('Task 3(b): shortcut is the only architectural difference within each pair')
plt.tight_layout()
plt.show()



**Task 3(b) interpretation - TODO:** Explain the measured effect of depth and shortcuts on optimization. Distinguish training difficulty from overfitting.

## Task 4 - Final CNN (optional)

Design a CNN with at most **500,000 parameters** and at most **20 training epochs**. Tune using validation data only. Report ablations covering at least three design choices, with controlled comparisons.

Before evaluating the test set, freeze the architecture, hyperparameters, checkpoint, and selection rule. Evaluate the final model on the test set once. Leave the following cell unexecuted until then.

In [ ]:
# %% Task 4: optional compact CNN and controlled ablations (validation only)
# This cell trains the final candidates; it never reads the test set.
assert 1 <= EPOCHS_T4 <= 20
FINAL_WIDTHS = (24, 48, 96)
# Depth 14, two blocks per stage. Four configurations isolate three design choices.
final_variants = {
    'full': {'residual': True, 'flip_crop': True, 'weight_decay': 5e-4},
    'no_shortcut': {'residual': False, 'flip_crop': True, 'weight_decay': 5e-4},
    'no_augmentation': {'residual': True, 'flip_crop': False, 'weight_decay': 5e-4},
    'no_weight_decay': {'residual': True, 'flip_crop': True, 'weight_decay': 0.0},
}
parameter_count = sum(p.numel() for p in CifarNetwork(14, True, FINAL_WIDTHS).parameters())
assert parameter_count <= 500_000, parameter_count
print(f'Final architecture: depth 14, widths {FINAL_WIDTHS}, {parameter_count:,} parameters')
task4_results = {}
for name, variant in final_variants.items():
    config = {'architecture': 'CifarNetwork', 'depth': 14, 'widths': list(FINAL_WIDTHS),
              'optimizer': 'Momentum', 'lr': 0.05, 'cosine': True,
              'epochs': EPOCHS_T4, **variant}
    data = augmented_dataset(train_indices, flip_crop=variant['flip_crop'])
    task4_results[name] = run_experiment(
        'task4_'+name, lambda r=variant['residual']: CifarNetwork(14, r, FINAL_WIDTHS),
        config, training_dataset=data)
comparison_table(task4_results)
plot_comparison(task4_results, 'Task 4: controlled design ablations')
complete_candidates = {name: r for name, r in task4_results.items() if r['status'] == 'complete'}
if not complete_candidates:
    raise RuntimeError('No completed Task 4 candidates; inspect training settings')
final_name = max(complete_candidates,
                 key=lambda name: complete_candidates[name]['history'][-1]['val_accuracy'])
final_run = complete_candidates[final_name]
print('Final choice (highest final validation accuracy):', final_name)
print('Checkpoint:', final_run['checkpoint'])
if task4_results['full']['status'] == 'complete':
    full_accuracy = task4_results['full']['history'][-1]['val_accuracy']
    show_table([[name, f"{100*(r['history'][-1]['val_accuracy']-full_accuracy):+.2f}"]
                for name, r in complete_candidates.items()],
               ['Variant', 'Validation change vs full (percentage points)'])



In [ ]:
# %% Task 4: one-time final test evaluation; deliberately requires opt-in
# Change to True ONLY after reviewing validation comparisons and freezing the choice.
RUN_FINAL_TEST = False
TEST_RECORD = OUTPUT_DIR / 'task4_final_test_once.json'
if TEST_RECORD.exists():
    saved_test = json.loads(TEST_RECORD.read_text())
    print('Previously saved test result (test set not evaluated again):')
    print(json.dumps(saved_test, indent=2))
elif not RUN_FINAL_TEST:
    print('Test set untouched. Review Task 4, then set RUN_FINAL_TEST=True to evaluate once.')
else:
    config = final_run['config']
    final_model = CifarNetwork(config['depth'], config['residual'], tuple(config['widths'])).to(DEVICE)
    state = torch.load(final_run['checkpoint'], map_location=DEVICE, weights_only=True)
    final_model.load_state_dict(state)
    assert sum(p.numel() for p in final_model.parameters()) <= 500_000
    assert config['epochs'] <= 20
    test_dataset = datasets.CIFAR10(root=DATA_DIR, train=False, download=True,
                                    transform=eval_transform)
    test_loader = make_loader(test_dataset)
    confusion = torch.zeros(10, 10, dtype=torch.int64)
    total_loss, total = 0.0, 0
    final_model.eval()
    with torch.no_grad():
        for x, y in test_loader:
            logits = final_model(x.to(DEVICE))
            total_loss += nn.functional.cross_entropy(logits, y.to(DEVICE), reduction='sum').item()
            predicted = logits.argmax(1).cpu()
            confusion += torch.bincount(y*10+predicted, minlength=100).reshape(10, 10)
            total += len(y)
    saved_test = {'selected_variant': final_name, 'config': config,
                  'checkpoint': final_run['checkpoint'], 'test_loss': total_loss/total,
                  'test_accuracy': confusion.diag().sum().item()/total,
                  'confusion_matrix': confusion.tolist()}
    atomic_json_save(saved_test, TEST_RECORD)
    print(f"Final test accuracy: {saved_test['test_accuracy']:.2%}")

# Display cached results without another inference pass over the test set.
if TEST_RECORD.exists():
    saved_test = json.loads(TEST_RECORD.read_text())
    matrix = np.asarray(saved_test['confusion_matrix'])
    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(matrix, cmap='Blues')
    ax.set(xticks=range(10), yticks=range(10), xlabel='Predicted', ylabel='True',
           title=f"Task 4: test accuracy {saved_test['test_accuracy']:.2%}")
    ax.set_xticklabels(raw_train.classes, rotation=45, ha='right')
    ax.set_yticklabels(raw_train.classes)
    fig.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.show()
    show_table([[name, f'{100*matrix[i,i]/matrix[i].sum():.2f}']
                for i, name in enumerate(raw_train.classes)], ['Class', 'Test accuracy (%)'])


**Task 4 interpretation - TODO:** Explain the design choices, ablation results, parameter budget, and final measured test accuracy.

## Submission checklist

- [ ] Name, index number, and GitHub profile included.
- [ ] Regular commits show progress over time.
- [ ] Tasks 1-3 complete; Task 4 clearly identified as optional.
- [ ] Experimental settings and measured results recorded.
- [ ] Important code, legible plots, comparison tables, and interpretations included.
- [ ] AI assistance acknowledgment updated accurately.
- [ ] Completed notebook checked for reproducibility.
- [ ] PDF export checked for clipped code, missing figures, and unreadable tables.
- [ ] PDF named `220381M_a02.pdf` .

The assignment states that only the PDF is used for grading; do not rely on external files to communicate essential results.